# Week 23 Optional Deep-Dive: AI Fairness 360 and LIME

> This notebook is OPTIONAL and self-paced. Finish the main Week 23 notebook
> (`week_23_ai_governance_ethics.ipynb`) first. Here we redo the same fairness
> audit and the same single-applicant explanation with a SECOND library each, so
> you see that the ideas are not tied to one tool.

## What you will do

1. **AI Fairness 360 (IBM)**: measure bias on the same credit model with
   `disparate_impact` and `statistical_parity_difference`, and map each metric
   back to the Fairlearn metric you already used. Then mitigate with `Reweighing`.
2. **LIME**: explain the SAME denied applicant you explained with SHAP in the main
   notebook, and compare the two explanations.
3. **Responsible AI Toolbox**: a short note on why this all-in-one dashboard is
   great but too heavy for a live class.

## Prerequisites

- Completed the main Week 23 notebook (Fairlearn + SHAP on the Adult credit model)

## Platform

Google Colab. CPU only, no AWS credentials. Same numpy below 2 pin and the same
plain install (no runtime restart) as the main notebook.

## Section 0: Setup and Rebuild the Credit Model

Same setup discipline as the main notebook: pin numpy below 2, then rebuild the
exact credit model so this notebook stands on its own. We use the same dataset,
the same `sex` protected attribute, the same `LogisticRegression`, and the same
`random_state=42` split, so every number here lines up with what you saw in the
main notebook.

Run the install cell first and let it run while you read; then continue down
the notebook to rebuild the model.

In [ ]:
# Install the libraries for this notebook. We pin numpy below 2 so the tooling
# installs cleanly. Base aif360 does NOT pull TensorFlow (the TF-based algorithms
# are an extra we do not use), so the install stays light.

!pip install -q \
    "numpy<2" \
    "scikit-learn>=1.4,<1.7" \
    "fairlearn>=0.12" \
    "aif360>=0.6" \
    "lime>=0.2" \
    "pandas>=2.0" \
    "matplotlib>=3.7"

In [ ]:
# This rebuilds the exact model from the main Week 23 notebook so the optional
# notebook is self-contained and the numbers match.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("numpy version:", np.__version__)  # expect 1.26.x (numpy<2)

# Load and reframe as loan approval (1 = approve), identical to the main notebook.
# Use the cached repo copy first; fall back to OpenML via Fairlearn if needed.
CACHE_URL = ("https://raw.githubusercontent.com/axel-sirota/"
             "bread-financial-academy/main/exercises/"
             "week_23_ai_governance_ethics/data/adult.csv")
try:
    full = pd.read_csv(CACHE_URL)
    X_raw = full.drop(columns=["__target__"]).copy()
    y = (full["__target__"] == ">50K").astype(int)
except Exception:
    from fairlearn.datasets import fetch_adult
    data = fetch_adult(as_frame=True)
    X_raw = data.data.copy()
    y = (data.target == ">50K").astype(int)
sensitive = X_raw["sex"]

numeric_features = ["age", "hours-per-week", "education-num"]
categorical_features = ["workclass", "marital-status", "occupation"]
model_features = numeric_features + categorical_features

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
credit_model = Pipeline(
    steps=[
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=1000)),
    ]
)

X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X_raw[model_features], y, sensitive, test_size=0.3, random_state=42
)
credit_model.fit(X_train, y_train)
y_pred = credit_model.predict(X_test)
print("Credit model rebuilt. Ready for AIF360 and LIME.")

## Section 1: AI Fairness 360 - a Second Opinion on Bias

In the main notebook Fairlearn told you the demographic parity difference.
AIF360 measures the same idea with different names and a different data
container. Seeing both teaches you that fairness metrics are concepts, not
library features.

The mapping you will confirm:

- AIF360 `statistical_parity_difference` is the same idea as Fairlearn's
  `demographic_parity_difference` (the gap in approval rates between groups). In
  AIF360 it is unprivileged minus privileged, so it is typically negative when
  the unprivileged group is approved less often.
- AIF360 `disparate_impact` is the RATIO of approval rates (unprivileged divided
  by privileged). The US "four-fifths rule" flags anything below 0.8.

AIF360 needs the data in its own `BinaryLabelDataset` container, with the
protected attribute encoded numerically and the privileged and unprivileged
groups named explicitly. That extra ceremony is one reason the main notebook used
Fairlearn, which works directly on pandas and sklearn output.

```python
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing
```

![AIF360: BinaryLabelDataset metrics](https://raw.githubusercontent.com/axel-sirota/bread-financial-academy/main/exercises/week_23_ai_governance_ethics/diagrams/aif360_flow.png)

In [ ]:
# DEMO: measure the same bias with AIF360 and compare names to Fairlearn.
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric

# AIF360 wants one dataframe with the features, the label, and a NUMERIC version
# of the protected attribute. Encode sex as 1 = Male (privileged), 0 = Female
# (unprivileged), purely so AIF360 can read it. This encoding choice is itself a
# values decision and worth noticing.
df = X_test[model_features].copy()
df = pd.get_dummies(df, columns=categorical_features)  # numeric features for AIF360
df["label"] = y_test.values
df["sex_code"] = (sens_test.values == "Male").astype(int)

privileged = [{"sex_code": 1}]
unprivileged = [{"sex_code": 0}]

# Build the AIF360 container from the pandas frame.
bld = BinaryLabelDataset(
    df=df,
    label_names=["label"],
    protected_attribute_names=["sex_code"],
    favorable_label=1,
    unfavorable_label=0,
)

metric = BinaryLabelDatasetMetric(
    bld, privileged_groups=privileged, unprivileged_groups=unprivileged
)

print("AIF360 disparate_impact (ratio, flag if < 0.8):",
      round(metric.disparate_impact(), 3))
print("AIF360 statistical_parity_difference (gap, same idea as Fairlearn DPD):",
      round(metric.statistical_parity_difference(), 3))
print("\nNotice: this is the SAME disparity Fairlearn reported in the main "
      "notebook, just expressed as a ratio and a signed gap.")

In [ ]:
# DEMO: mitigate bias with AIF360's Reweighing.
# Reweighing does NOT change any feature or label. It only assigns a weight to
# each training row so that, after weighting, the protected groups are balanced.
# You then train the model with sample_weight.
from aif360.algorithms.preprocessing import Reweighing

# Build a BinaryLabelDataset for the TRAINING data the same way.
df_tr = X_train[model_features].copy()
df_tr = pd.get_dummies(df_tr, columns=categorical_features)
df_tr["label"] = y_train.values
df_tr["sex_code"] = (sens_train.values == "Male").astype(int)

bld_tr = BinaryLabelDataset(
    df=df_tr,
    label_names=["label"],
    protected_attribute_names=["sex_code"],
    favorable_label=1,
    unfavorable_label=0,
)

rw = Reweighing(unprivileged_groups=unprivileged, privileged_groups=privileged)
bld_tr_rw = rw.fit_transform(bld_tr)

# The new per-row weights are what reduce the disparity at training time.
sample_weights = bld_tr_rw.instance_weights
print("Reweighing produced per-row training weights.")
print("Example weights (first 5):", np.round(sample_weights[:5], 3))
print("\nYou would now retrain the model with these weights:")
print("    model.fit(X_train, y_train, clf__sample_weight=sample_weights)")
print("Reweighing fixes bias in the DATA stage; ThresholdOptimizer (main "
      "notebook) fixed it in the PREDICTION stage. Two different stages, same "
      "goal.")

### Think About It

You measured the same disparity with two libraries and mitigated it at two
different stages.

- Reweighing changes the training data stage; the main notebook's
  `ThresholdOptimizer` changed the prediction stage. If a regulator asked "where
  in the pipeline did you correct for fairness", how would your answer differ
  between the two approaches, and which is easier to audit after the fact?
- AIF360 made you encode sex as a 1/0 privileged/unprivileged variable. Who
  decides which group is "privileged", and how could that labeling choice itself
  carry bias?

### Optional try-it

Re-run the AIF360 metrics in the demo using `race` as the protected attribute
instead of `sex` (build `race_code` for one race group vs the rest). Is the
disparate impact better or worse than for sex?

## Section 2: LIME - a Different Lens on Local Explanations

In the main notebook, SHAP explained a single denied applicant with exact feature
contributions. LIME answers the same "why this decision" question with different
math: it perturbs the applicant's features many times, watches how the model's
prediction changes, and fits a simple local model around that one point. The
weights of that simple local model are the explanation.

Two different methods, one question. Where they AGREE on the top factors, you can
be more confident in an adverse-action notice. Where they DISAGREE, that is a
signal to look closer before you put a reason in writing to a customer.

```python
from lime.lime_tabular import LimeTabularExplainer
explainer = LimeTabularExplainer(training_array, feature_names=..., mode="classification")
explanation = explainer.explain_instance(one_row, model.predict_proba)
```

![LIME vs SHAP local explanations](https://raw.githubusercontent.com/axel-sirota/bread-financial-academy/main/exercises/week_23_ai_governance_ethics/diagrams/lime_flow.png)

In [ ]:
# DEMO: explain the same denied applicant with LIME.
from lime.lime_tabular import LimeTabularExplainer

# Like SHAP in the main notebook, LIME works on the ENCODED numeric data, so we
# explain the LogisticRegression step using the fitted preprocessing.
prep = credit_model.named_steps["prep"]
clf = credit_model.named_steps["clf"]

X_train_enc = prep.transform(X_train).toarray()
X_test_enc = prep.transform(X_test).toarray()
feature_names = list(prep.get_feature_names_out())

# Find the same kind of denied applicant the main notebook explained.
denied_idx = int(np.where(y_pred == 0)[0][0])

explainer = LimeTabularExplainer(
    training_data=X_train_enc,
    feature_names=feature_names,
    class_names=["deny", "approve"],
    mode="classification",
)

# explain_instance needs the row and a predict_proba function over encoded rows.
explanation = explainer.explain_instance(
    data_row=X_test_enc[denied_idx],
    predict_fn=clf.predict_proba,
    num_features=8,
)

print("LIME explanation for the denied applicant (feature, weight):")
for feat, weight in explanation.as_list():
    print(f"  {feat}: {round(weight, 4)}")

In [ ]:
# DEMO: line up the LIME top factors against what SHAP would say for the same row.
# LIME weights below 0 push toward "deny". Pull the strongest deny-drivers.
lime_pairs = explanation.as_list()
lime_deny_drivers = sorted(lime_pairs, key=lambda kv: kv[1])[:3]
lime_top = [feat for feat, _ in lime_deny_drivers]

print("LIME top denial factors:", lime_top)
print("\nCompare these to the SHAP top denial factors you printed in the main "
      "notebook's Lab 2 for the same applicant.")
print("Where LIME and SHAP agree, that factor is a strong candidate for the "
      "adverse-action notice. Where they disagree, investigate before writing it "
      "down.")

# Optional: a quick bar chart of the LIME weights for a visual.
feats = [f for f, _ in lime_pairs]
weights = [w for _, w in lime_pairs]
colors = ["tab:red" if w < 0 else "tab:blue" for w in weights]
plt.barh(feats, weights, color=colors)
plt.title("LIME local explanation - denied applicant (red = toward deny)")
plt.tight_layout()
plt.show()

## Section 3: Microsoft Responsible AI Toolbox - Powerful, but Heavy for Class

Microsoft's Responsible AI Toolbox (the `raiwidgets` package) bundles error
analysis, fairness assessment, and explainability into a single interactive
dashboard you drive from one `RAIInsights` object. It is excellent for a deep,
exploratory review of a model: you can slice errors by cohort, inspect fairness
across groups, and drill into feature importance all in one place.

We do not run it live in this class for one practical reason: its interactive
widget stack is heavy and does not render reliably inside a time-boxed Colab
session. Restart loops and widget-rendering issues eat the very minutes a 90
minute class does not have.

What this means for you:
- For a quick, scriptable audit in a notebook, Fairlearn plus SHAP (the main
  notebook) is the lighter, more reliable choice.
- For a thorough, interactive model review on a workstation or a dedicated
  environment, the Responsible AI Toolbox dashboard is worth learning on your own
  time.

Self-paced pointer: search the official "Responsible AI Toolbox" documentation
and the `responsibleai` / `raiwidgets` packages, and try the dashboard on this
same Adult credit model when you are not on the clock.

## Wrap-up: One Idea, Many Tools

You just confirmed that responsible AI is about concepts, not a single library:

- **Fairness**: AIF360's `disparate_impact` and `statistical_parity_difference`
  are the same disparities Fairlearn reported, and AIF360's `Reweighing` fixes
  bias at the data stage where the main notebook's `ThresholdOptimizer` fixed it
  at the prediction stage.
- **Explainability**: LIME and SHAP both answered "why was this applicant
  denied", with different math. Agreement between them strengthens an
  adverse-action notice; disagreement is a flag to investigate.
- **Tooling tradeoffs**: lighter scriptable tools (Fairlearn, SHAP, LIME) for
  notebooks and CI; the heavier Responsible AI Toolbox dashboard for deep
  interactive reviews off the clock.

### Optional try-it recap

1. Redo the AIF360 metrics with `race` as the protected attribute.
2. Run LIME on an APPROVED applicant and contrast it with the denied one.

Back to the main track: in Week 24 (Capstone), add a fairness audit and a local
explanation to your own model. You now have two libraries for each, so you can
pick the right tool and defend the choice.